# 데이터 확인

`01_proposal.md` 작성 테스트 / 검증 기록

- 데이터: SEC Form 13F (2022Q3, 2023Q4 분기 zip)
- 원본 zip 은 용량이 커서 저장소에 없음. 아래 명령으로 먼저 받아야 함.
  ```sh
  uv run src/probe_data.py   # probe_out/raw/ 에 zip 생성
  ```
- 필요 라이브러리: pandas (zipfile 은 표준 라이브러리)

In [1]:
import zipfile
import pandas as pd

# zip 파일 경로. 이 노트북이 있는 01_proposal 폴더 기준 상대경로.
zip_2023q4 = "../probe_out/raw/2023q4_form13f.zip"
zip_2022q3 = "../probe_out/raw/2022q3_form13f.zip"

# zip 안의 표(tsv) 하나를 읽어서 데이터프레임으로 돌려주는 함수.
# dtype=str: 모든 값을 일단 문자열로 읽음. 숫자 변환은 필요할 때 직접 함.
# usecols: 필요한 컬럼만 골라 읽어서 메모리를 아낌. None 이면 전체.
def read_table(zip_path, filename, usecols=None):
    zf = zipfile.ZipFile(zip_path)                # zip 파일 열기
    f = zf.open(filename)                          # zip 안의 파일 하나 열기
    df = pd.read_csv(f, sep="\t", dtype=str, usecols=usecols)  # 탭으로 나뉜 표 읽기
    zf.close()
    return df

print("준비 완료")

준비 완료


In [2]:
# read_table 함수가 잘 동작하는지 작은 표(SUBMISSION)로 확인.
# 인자 순서 주의: read_table(zip경로, zip안의파일이름)
test = read_table(zip_2023q4, "SUBMISSION.tsv")
print("행 수:", len(test))
print("열 이름:", list(test.columns))
test.head()   # 앞 5줄 미리보기

행 수: 9196
열 이름: ['ACCESSION_NUMBER', 'FILING_DATE', 'SUBMISSIONTYPE', 'CIK', 'PERIODOFREPORT']


,ACCESSION_NUMBER,FILING_DATE,SUBMISSIONTYPE,CIK,PERIODOFREPORT
0,0001025835-23-000086,31-OCT-2023,13F-HR,0001025835,30-SEP-2023
1,0000905148-23-001261,31-OCT-2023,13F-HR/A,0000931588,31-MAR-2022
2,0001104659-23-112948,31-OCT-2023,13F-HR,0001686444,30-SEP-2023
3,0001172661-23-003551,31-OCT-2023,13F-HR,0000883677,30-SEP-2023
4,0001085146-23-003989,31-OCT-2023,13F-HR,0001730456,30-SEP-2023


## 1. 파일 구조 (기획서 6절)

zip 안에 분석용 tsv 테이블 7개가 있는지 확인. (메타데이터-리드미 파일 2개는 별도)

In [3]:
# 2023Q4 zip 안에 들어있는 파일 이름을 하나씩 출력.
zf = zipfile.ZipFile(zip_2023q4)
for name in zf.namelist():
    print(name)
zf.close()

COVERPAGE.tsv
INFOTABLE.tsv
OTHERMANAGER.tsv
OTHERMANAGER2.tsv
SIGNATURE.tsv
SUBMISSION.tsv
SUMMARYPAGE.tsv
FORM13F_metadata.json
FORM13F_readme.htm


## 2. INFOTABLE

기획서 값: 2023Q4 INFOTABLE = 2,886,468행 x 15열, CUSIP 고유값 35,442개.

In [4]:
# INFOTABLE 읽기. 뒤 절에서 쓸 5개 컬럼만 골라 읽어 메모리를 아낌.
info = read_table(zip_2023q4, "INFOTABLE.tsv",
                  usecols=["ACCESSION_NUMBER", "NAMEOFISSUER", "CUSIP", "VALUE", "SSHPRNAMT"])

# 전체 열 개수는 첫 줄(제목 줄)만 읽어서 셈. nrows=0 이면 데이터는 안 읽고 열 이름만 가져옴.
zf = zipfile.ZipFile(zip_2023q4)
header = pd.read_csv(zf.open("INFOTABLE.tsv"), sep="\t", nrows=0)
zf.close()

print("행 수:", len(info))
print("열 수:", len(header.columns))
print("CUSIP 종류 수:", info["CUSIP"].nunique())

행 수: 2886468
열 수: 15
CUSIP 종류 수: 35442


## 3. VALUE 단위 변경

주당 단가 = VALUE / SSHPRNAMT. 단위가 천달러면 이 값이 아주 작게 나옴.
기획서 값: 2022Q3 중앙값 0.0542 (천달러), 2023Q4 47.56 (달러), 약 880배 차이.

In [5]:
# 주당 단가의 중앙값을 구하는 함수.
def median_price(df):
    value = pd.to_numeric(df["VALUE"], errors="coerce")       # 문자열을 숫자로 (안되면 NaN)
    shares = pd.to_numeric(df["SSHPRNAMT"], errors="coerce")
    price = value / shares                                     # 주당 단가
    price = price[price > 0]              # 0 이하 제거
    price = price[price < 10000000]       # 비정상적으로 큰 값 제거
    return price.median()

# 2022Q3 는 단가 계산에 필요한 2개 컬럼만 읽음.
info_2022q3 = read_table(zip_2022q3, "INFOTABLE.tsv", usecols=["VALUE", "SSHPRNAMT"])

price_2022q3 = median_price(info_2022q3)
price_2023q4 = median_price(info)

print("2022Q3 중앙 단가:", round(price_2022q3, 4))
print("2023Q4 중앙 단가:", round(price_2023q4, 4))
print("배율:", round(price_2023q4 / price_2022q3), "배")

2022Q3 중앙 단가: 0.0542
2023Q4 중앙 단가: 47.5599
배율: 878 배


## 4. 공시 지연

제출일(FILING_DATE) - 보고기준일(PERIODOFREPORT) = 공시까지 걸린 일수.
기획서 값: 중앙값 44일, 45일 초과 9.0%.

In [6]:
sub = read_table(zip_2023q4, "SUBMISSION.tsv", usecols=["FILING_DATE", "PERIODOFREPORT"])

# 날짜 형식이 '31-OCT-2023' (일-월-년) 이라 그 형식을 지정해서 날짜로 변환.
filing = pd.to_datetime(sub["FILING_DATE"], format="%d-%b-%Y")
period = pd.to_datetime(sub["PERIODOFREPORT"], format="%d-%b-%Y")

delay = (filing - period).dt.days     # 두 날짜의 차이를 일수로

print("제출 건수:", len(delay))
print("중앙값(일):", delay.median())
print("75% 지점(일):", delay.quantile(0.75))
print("최대(일):", delay.max())

over_45 = (delay > 45).mean() * 100   # 45일을 넘긴 제출의 비율(%)
print("45일 초과 비율(%):", round(over_45, 1))

제출 건수: 9196
중앙값(일): 44.0
75% 지점(일): 45.0
최대(일): 2776
45일 초과 비율(%): 9.0


## 5. 수정보고서 / 비밀보유 해제 / 운용사 수

기획서 값: 수정보고서 3.4%, 비밀보유 해제 1.1%, 운용사 고유값 8,604개.

In [7]:
cov = read_table(zip_2023q4, "COVERPAGE.tsv",
                 usecols=["FILINGMANAGER_NAME", "AMENDMENTTYPE", "CONFDENIEDEXPIRED"])

# notna() = 값이 채워진 칸(True/False). mean() 으로 비율을 구함.
amend_pct = cov["AMENDMENTTYPE"].notna().mean() * 100
conf_pct = cov["CONFDENIEDEXPIRED"].notna().mean() * 100

print("수정보고서 비율(%):", round(amend_pct, 1))
print("비밀보유 해제 비율(%):", round(conf_pct, 1))
print("운용사 종류 수:", cov["FILINGMANAGER_NAME"].nunique())

수정보고서 비율(%): 3.4
비밀보유 해제 비율(%): 1.1
운용사 종류 수: 8604


## 6. 문자열 품질 문제

`CUSIP` 대소문자 혼재, `000000000` 플레이스홀더 등이 실제로 있는지 확인.

In [8]:
cusip = info["CUSIP"].dropna()

# CUSIP 은 원래 대문자. 대문자로 바꾼 값이 원래와 다르면 소문자가 섞여 있던 것.
has_lower = cusip != cusip.str.upper()

print("소문자 섞인 CUSIP 개수:", has_lower.sum())
print("000000000 (빈 코드) 개수:", (cusip == "000000000").sum())
print("소문자 CUSIP 예시:", list(cusip[has_lower].head(3)))

소문자 섞인 CUSIP 개수: 11650
000000000 (빈 코드) 개수: 106
소문자 CUSIP 예시: ['00846u101', '02209s103', '00206r102']


## 결론

위 출력값이 `01_proposal.md` 수치와 일치함.
데이터 규모, 단위 변경, 공시 지연, 수정보고서 비율 확인